# Region Analysis — MIBI_TNBC

Unsupervised tissue region discovery applied to ~40 TNBC patients.

**Three experiments:**
1. **Pooled region discovery** — all patients together, k-means sweep, spatial maps
2. **Cross-patient reproducibility** — cluster each patient independently, match clusters by composition similarity, quantify consistency with Jensen-Shannon divergence
3. **CIM vs EarlyFusion32 comparison** — same patches embedded by each model, compare compartment distinctiveness

**Model used for experiments 1 & 2:** `MIBI_TNBC_CIM_VICReg`  
**Models used for experiment 3:** `MIBI_TNBC_CIM_VICReg` vs `MIBI_TNBC_EarlyFusion32_VICReg`

In [ ]:
# ── CONFIGURATION ──────────────────────────────────────────────────────────

CIM_WORK_DIR   = '/home/simon_g/isilon_images_mnt/10_MetaSystems/MetaSystemsData/_simon/src/MCA/z_RUNS/MIBI_TNBC_CIM_VICReg'
EF_WORK_DIR    = '/home/simon_g/isilon_images_mnt/10_MetaSystems/MetaSystemsData/_simon/src/MCA/z_RUNS/MIBI_TNBC_EarlyFusion32_VICReg'

H5_PATH        = '/home/simon_g/isilon_images_mnt/10_MetaSystems/MetaSystemsData/_simon/data/MCI_data/h5_files/MIBI_TNBC/MIBI_TNBC.h5'
MARKERS_PATH   = '/home/simon_g/isilon_images_mnt/10_MetaSystems/MetaSystemsData/_simon/data/MCI_data/h5_files/MIBI_TNBC/used_markers.txt'

REGION_PATCH_SIZE = 64    # spatial size of each region patch in pixels
STRIDE            = REGION_PATCH_SIZE // 2   # 50 % overlap

N_CLUSTERS_LIST   = [2, 4, 6, 8, 10]   # k values to explore
K_REPRO           = 6                   # k used for cross-patient reproducibility

PCA_COMPONENTS    = 64

_REF_PATCH = 64
_REF_BATCH = 64
BATCH_SIZE  = max(1, int(_REF_BATCH * (_REF_PATCH / REGION_PATCH_SIZE) ** 2))

UMAP_MAX_SAMPLES  = 50_000
N_JOBS            = 8

# Representative patients to show in side-by-side spatial maps (experiment 3).
# Leave as None to auto-pick 3 patients with the most patches.
SHOW_PATIENTS     = None
N_SHOW_PATIENTS   = 3

SAVE_DIR = f'../z_RUNS/region_analysis_MIBI_TNBC/ps{REGION_PATCH_SIZE}'

# ───────────────────────────────────────────────────────────────────────────
print(f'Patch size : {REGION_PATCH_SIZE}px   Batch size : {BATCH_SIZE}')

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

for _var in ('OMP_NUM_THREADS', 'MKL_NUM_THREADS', 'OPENBLAS_NUM_THREADS',
             'NUMEXPR_NUM_THREADS', 'VECLIB_MAXIMUM_THREADS'):
    os.environ[_var] = str(N_JOBS)

import warnings
warnings.filterwarnings('ignore')

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.gridspec as gridspec

import numpy as np
import h5py
import torch
import torch.nn.functional as F
import pandas as pd
from pathlib import Path
from tqdm import tqdm
from collections import Counter, defaultdict

from sklearn.cluster import MiniBatchKMeans
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score
from scipy.optimize import linear_sum_assignment
from scipy.spatial.distance import jensenshannon
import umap
import json

from threadpoolctl import threadpool_limits
threadpool_limits(N_JOBS)

from MCA.src.utils import load_checkpoint

SAVE_DIR = Path(SAVE_DIR)
SAVE_DIR.mkdir(parents=True, exist_ok=True)

DEVICE = 'cuda:0' if torch.cuda.is_available() else 'cpu'
print(f'Device  : {DEVICE}')
print(f'Output  : {SAVE_DIR.resolve()}')

## 1 · Load backbones

In [ ]:
def load_backbone(work_dir, device):
    result   = load_checkpoint(work_dir, device=device)
    backbone = result['model'].backbone
    backbone.eval()
    return backbone

cim_backbone = load_backbone(CIM_WORK_DIR, DEVICE)
ef_backbone  = load_backbone(EF_WORK_DIR,  DEVICE)

# Measure feature dims from a dummy forward pass
with torch.no_grad():
    dummy = torch.zeros(1, 37, REGION_PATCH_SIZE, REGION_PATCH_SIZE, device=DEVICE)
    CIM_DIM = cim_backbone(dummy)[0].squeeze(-1).squeeze(-1).shape[1]
    EF_DIM  = ef_backbone(dummy)[0].squeeze(-1).squeeze(-1).shape[1]

print(f'CIM feature dim       : {CIM_DIM}')
print(f'EarlyFusion32 feat dim: {EF_DIM}')

## 2 · Load HDF5 metadata

In [ ]:
def decode(arr):
    return np.array([x.decode() if isinstance(x, bytes) else x for x in arr])

with h5py.File(H5_PATH, 'r') as f:
    all_marker_names = decode(f['marker_names'][:])
    cell_sample_ids  = decode(f['coords']['sample_id'][:])
    cell_dim1        = f['coords']['DIM1'][:].astype(int)
    cell_dim2        = f['coords']['DIM2'][:].astype(int)
    cell_annotations = decode(f['annotation'][:])
    unique_samples   = decode(f['sample_ids'][:])

with open(MARKERS_PATH) as fh:
    used_names = np.array([l.strip() for l in fh if l.strip()])

marker2idx     = {m: i for i, m in enumerate(all_marker_names)}
marker_indices = np.array([marker2idx[m] for m in used_names])

IGNORE = {'Unidentified'}
cell_mask  = np.array([a not in IGNORE for a in cell_annotations])

print(f'Markers in panel : {len(all_marker_names)}  |  Used: {len(marker_indices)}')
print(f'Cells in dataset : {len(cell_dim1):,}  (after ignore: {cell_mask.sum():,})')
print(f'Patients ({len(unique_samples)}): {list(unique_samples[:5])} ...')

## 3 · Precompute sliding-window grid

In [ ]:
ps       = REGION_PATCH_SIZE
all_meta = []   # (sample_id, y, x, img_H, img_W)

with h5py.File(H5_PATH, 'r') as f:
    for sid in unique_samples:
        H, W = f['data'][sid]['image'].shape[:2]
        for y in range(0, H - ps + 1, STRIDE):
            for x in range(0, W - ps + 1, STRIDE):
                all_meta.append((sid, y, x, H, W))

# Per-patient patch counts
sample_patch_counts = Counter(m[0] for m in all_meta)
sample_dims         = {}
for sid, y, x, H, W in all_meta:
    if sid not in sample_dims:
        sample_dims[sid] = (H, W)

# Choose representative patients for visualisation
if SHOW_PATIENTS is None:
    SHOW_PATIENTS = [sid for sid, _ in sample_patch_counts.most_common(N_SHOW_PATIENTS)]

print(f'Total patches : {len(all_meta):,}')
print(f'Per-patient  : min={min(sample_patch_counts.values())},'
      f' max={max(sample_patch_counts.values())},'
      f' mean={int(np.mean(list(sample_patch_counts.values())))}')
print(f'Show patients : {SHOW_PATIENTS}')

## 4 · Embed all patches — CIM and EarlyFusion32

One sequential HDF5 read per patient. Embeddings saved as float16 to save RAM.

In [ ]:
# Build patch index and per-patient index lists
patch_index            = {(sid, y, x): i for i, (sid, y, x, H, W) in enumerate(all_meta)}
sample_to_meta_indices = defaultdict(list)
for i, (sid, *_) in enumerate(all_meta):
    sample_to_meta_indices[sid].append(i)

@torch.no_grad()
def flush_batch(batch_list, backbone):
    device = next(backbone.parameters()).device
    t      = torch.from_numpy(np.array(batch_list, dtype=np.float32)).to(device)
    feats  = backbone(t)
    if isinstance(feats, (tuple, list)):
        feats = feats[0]
    feats = feats.squeeze(-1).squeeze(-1)
    return F.normalize(feats, dim=1).cpu().numpy().astype(np.float16)


def embed_all(backbone, feat_dim, label='model'):
    embeddings = np.zeros((len(all_meta), feat_dim), dtype=np.float16)
    with h5py.File(H5_PATH, 'r') as f:
        pbar = tqdm(total=len(all_meta), desc=f'Embedding [{label}]', unit='patch', dynamic_ncols=True)
        for sid in unique_samples:
            image   = f['data'][sid]['image'][:].astype(np.float32)[:, :, marker_indices]
            indices = sample_to_meta_indices[sid]
            batch_patches, batch_idx = [], []
            for i in indices:
                _, y, x, H, W = all_meta[i]
                batch_patches.append(image[y:y+ps, x:x+ps, :].transpose(2, 0, 1))
                batch_idx.append(i)
                if len(batch_patches) == BATCH_SIZE:
                    embeddings[batch_idx] = flush_batch(batch_patches, backbone)
                    pbar.update(len(batch_patches))
                    batch_patches, batch_idx = [], []
            if batch_patches:
                embeddings[batch_idx] = flush_batch(batch_patches, backbone)
                pbar.update(len(batch_patches))
        pbar.close()
    print(f'[{label}] Embeddings: {embeddings.shape}  {embeddings.nbytes/1024**2:.0f} MB')
    return embeddings


cim_emb = embed_all(cim_backbone, CIM_DIM, label='CIM')
ef_emb  = embed_all(ef_backbone,  EF_DIM,  label='EarlyFusion32')

## 5 · Map cells to patches (once, shared by both models)

In [ ]:
patch_to_cell_annotations = defaultdict(list)

for cidx in tqdm(range(len(cell_sample_ids)), desc='Mapping cells to patches', unit='cell', dynamic_ncols=True):
    if not cell_mask[cidx]:
        continue
    sid = cell_sample_ids[cidx]
    if sid not in sample_dims:
        continue
    cy, cx     = int(cell_dim1[cidx]), int(cell_dim2[cidx])
    H, W       = sample_dims[sid]
    annotation = cell_annotations[cidx]

    py_start = (max(0, cy - ps + 1) // STRIDE) * STRIDE
    px_start = (max(0, cx - ps + 1) // STRIDE) * STRIDE
    for py in range(py_start, min(cy + 1, H - ps + 1), STRIDE):
        for px in range(px_start, min(cx + 1, W - ps + 1), STRIDE):
            pidx = patch_index.get((sid, py, px))
            if pidx is not None:
                patch_to_cell_annotations[pidx].append(annotation)

all_cell_types = sorted({a for anns in patch_to_cell_annotations.values() for a in anns})
print(f'Mapped {cell_mask.sum():,} cells to {len(patch_to_cell_annotations):,} patches')
print(f'Cell types: {all_cell_types}')

## 6 · PCA + k sweep (CIM, pooled)

Fit PCA on all patients pooled — shared embedding space needed for cross-patient analysis.

In [ ]:
pca_cim     = PCA(n_components=PCA_COMPONENTS, random_state=42)
cim_emb_pca = pca_cim.fit_transform(cim_emb.astype(np.float32))
print(f'CIM PCA explained variance: {pca_cim.explained_variance_ratio_.sum():.1%}')

# k sweep on a subsample
idx_sub = np.random.RandomState(42).choice(len(cim_emb_pca), min(15_000, len(cim_emb_pca)), replace=False)
X_sub   = cim_emb_pca[idx_sub]

inertia_cim, sil_cim = [], []
for k in tqdm(N_CLUSTERS_LIST, desc='k sweep (CIM)'):
    km = MiniBatchKMeans(n_clusters=k, random_state=42, n_init=5, batch_size=2048)
    km.fit(X_sub)
    inertia_cim.append(km.inertia_)
    sil_cim.append(silhouette_score(X_sub, km.labels_, metric='euclidean', sample_size=5000))

best_k = N_CLUSTERS_LIST[int(np.argmax(sil_cim))]
print(f'Best silhouette: k={best_k}  ({max(sil_cim):.3f})')

fig, axes = plt.subplots(1, 2, figsize=(10, 3))
axes[0].plot(N_CLUSTERS_LIST, inertia_cim, 'o-')
axes[0].set(xlabel='k', ylabel='Inertia', title='Elbow — CIM')
axes[1].plot(N_CLUSTERS_LIST, sil_cim, 'o-')
axes[1].axvline(best_k, color='red', linestyle='--', alpha=0.6, label=f'best k={best_k}')
axes[1].set(xlabel='k', ylabel='Silhouette', title='Silhouette — CIM')
axes[1].legend()
for ax in axes: ax.set_xticks(N_CLUSTERS_LIST)
plt.tight_layout()
plt.savefig(SAVE_DIR / 'k_sweep_CIM.png', dpi=150, bbox_inches='tight')
plt.show()

## 7 · Pooled clustering + UMAP + spatial maps (CIM, best k)

In [ ]:
def composition_from_labels(labels, all_meta, patch_to_cell_annotations, all_cell_types, k):
    """Returns (k, n_cell_types) composition matrix (rows sum to 1)."""
    cluster_counts = defaultdict(Counter)
    for pidx, ann_list in patch_to_cell_annotations.items():
        cl = int(labels[pidx])
        cluster_counts[cl].update(ann_list)
    comp = np.zeros((k, len(all_cell_types)))
    for cl in range(k):
        total = sum(cluster_counts[cl].values())
        if total:
            for j, ct in enumerate(all_cell_types):
                comp[cl, j] = cluster_counts[cl].get(ct, 0) / total
    return comp, cluster_counts


def run_pooled(k, emb_pca, cmap_name='tab10'):
    out_dir = SAVE_DIR / f'k_{k}'
    out_dir.mkdir(exist_ok=True)
    cmap = plt.cm.get_cmap(cmap_name, k)

    km = MiniBatchKMeans(n_clusters=k, random_state=42, n_init=10, batch_size=4096, max_iter=300)
    labels = km.fit_predict(emb_pca)
    print(f'k={k}  cluster sizes: {np.bincount(labels).tolist()}')

    # UMAP
    n_u    = min(UMAP_MAX_SAMPLES, len(emb_pca))
    idx_u  = np.random.RandomState(42).choice(len(emb_pca), n_u, replace=False)
    reducer = umap.UMAP(n_components=2, n_neighbors=30, min_dist=0.1, metric='cosine',
                        random_state=42, n_jobs=N_JOBS, verbose=False)
    umap_xy = reducer.fit_transform(emb_pca[idx_u].astype(np.float32))

    fig, ax = plt.subplots(figsize=(8, 6))
    sc = ax.scatter(umap_xy[:, 0], umap_xy[:, 1],
                    c=labels[idx_u], cmap=cmap, vmin=-0.5, vmax=k-0.5, s=2, alpha=0.5)
    plt.colorbar(sc, ax=ax, label='Cluster', ticks=range(k))
    ax.set(title=f'UMAP  CIM + VICReg  MIBI_TNBC  k={k}', xlabel='UMAP 1', ylabel='UMAP 2')
    plt.tight_layout()
    plt.savefig(out_dir / 'umap_CIM.png', dpi=150, bbox_inches='tight')
    plt.show()

    # Spatial maps — show SHOW_PATIENTS side by side
    fig, axes = plt.subplots(1, len(SHOW_PATIENTS), figsize=(7*len(SHOW_PATIENTS), 6), squeeze=False)
    for ax, sid in zip(axes[0], SHOW_PATIENTS):
        H, W     = sample_dims[sid]
        canvas   = np.full((H, W, 4), [0.85, 0.85, 0.85, 1.0])
        for i in sample_to_meta_indices[sid]:
            _, y, x, _, _ = all_meta[i]
            canvas[y:y+ps, x:x+ps] = np.array(cmap(int(labels[i])))
        ax.imshow(canvas)
        ax.set_title(f'Patient {sid}', fontsize=10)
        ax.axis('off')
    handles = [mpatches.Patch(color=cmap(c), label=f'C{c}') for c in range(k)]
    fig.legend(handles=handles, loc='lower center', ncol=k, frameon=False)
    fig.suptitle(f'Spatial region map — CIM + VICReg, MIBI_TNBC, k={k}', fontsize=12)
    plt.tight_layout(rect=[0, 0.04, 1, 1])
    plt.savefig(out_dir / 'spatial_map_CIM.png', dpi=150, bbox_inches='tight')
    plt.show()

    # Composition bar chart
    comp, cc = composition_from_labels(labels, all_meta, patch_to_cell_annotations, all_cell_types, k)
    comp_df  = pd.DataFrame(comp, columns=all_cell_types)
    fig, ax  = plt.subplots(figsize=(12, 5))
    comp_df.plot(kind='bar', stacked=True, ax=ax, colormap='tab20', width=0.8)
    ax.set(xlabel='Cluster', ylabel='Cell-type fraction',
           title=f'Cell-type composition — CIM, MIBI_TNBC, k={k}')
    ax.legend(bbox_to_anchor=(1.01, 1), loc='upper left', fontsize=8, frameon=False)
    ax.set_xticklabels([f'C{c}' for c in range(k)], rotation=0)
    plt.tight_layout()
    plt.savefig(out_dir / 'composition_CIM.png', dpi=150, bbox_inches='tight')
    plt.show()

    print(f'\n  {"Cluster":<10} {"Top cell type":<28} {"Frac":>6}  {"N patches":>10}')
    print('  ' + '-'*60)
    for cl in range(k):
        dom = all_cell_types[comp[cl].argmax()]
        print(f'  C{cl:<9} {dom:<28} {comp[cl].max():>5.1%}  {int(np.sum(labels==cl)):>10,}')

    return labels, comp, comp_df


cim_labels_pooled, cim_comp_pooled, cim_comp_df = run_pooled(K_REPRO, cim_emb_pca)

---
## Experiment 1 — Cross-patient reproducibility

**Method:**  
1. Cluster each patient's patches independently (k = K_REPRO) in the shared PCA space
2. For every pair of patients, find the optimal cluster matching via the Hungarian algorithm on Jensen-Shannon divergence between cluster composition vectors
3. Report the JS divergence of matched clusters: low = same biological compartment recovered in both patients
4. Visualise as a heatmap of matched compositions across all patients

**Expected result:** CIM embeddings produce reproducible compartments (tumour nests, stroma, immune infiltrate) across held-out patients.

In [ ]:
def patient_composition(sid, labels_all, k):
    """Extract per-cluster composition vector for one patient."""
    pid_indices = sample_to_meta_indices[sid]
    counts = defaultdict(Counter)
    for i in pid_indices:
        anns = patch_to_cell_annotations.get(i, [])
        counts[int(labels_all[i])].update(anns)
    comp = np.zeros((k, len(all_cell_types)))
    for cl in range(k):
        total = sum(counts[cl].values())
        if total:
            for j, ct in enumerate(all_cell_types):
                comp[cl, j] = counts[cl].get(ct, 0) / total
    return comp  # (k, n_cell_types)


def per_patient_cluster(sid, emb_pca, k):
    """Cluster one patient's patches independently, return labels for ALL patches (others==-1)."""
    idx = sample_to_meta_indices[sid]
    X   = emb_pca[idx]
    if len(X) < k:
        return None, None
    km     = MiniBatchKMeans(n_clusters=k, random_state=42, n_init=10, batch_size=min(2048, len(X)))
    local_labels = km.fit_predict(X)
    # Build composition vector per cluster for this patient
    counts = defaultdict(Counter)
    for li, gi in enumerate(idx):
        anns = patch_to_cell_annotations.get(gi, [])
        counts[int(local_labels[li])].update(anns)
    comp = np.zeros((k, len(all_cell_types)))
    for cl in range(k):
        total = sum(counts[cl].values())
        if total:
            for j, ct in enumerate(all_cell_types):
                comp[cl, j] = counts[cl].get(ct, 0) / total
    return local_labels, comp  # local_labels: (n_patches_for_patient,), comp: (k, n_types)


def match_clusters(comp_a, comp_b):
    """Hungarian matching: returns (matched_b_for_each_a, cost_matrix)."""
    k = len(comp_a)
    cost = np.zeros((k, k))
    for i in range(k):
        for j in range(k):
            p = comp_a[i] + 1e-9
            q = comp_b[j] + 1e-9
            p /= p.sum(); q /= q.sum()
            cost[i, j] = jensenshannon(p, q)
    row_ind, col_ind = linear_sum_assignment(cost)
    return col_ind, cost[row_ind, col_ind]  # matched_js per cluster


print(f'Clustering {len(unique_samples)} patients independently (k={K_REPRO})...')
per_patient_comps = {}    # sid -> (k, n_types) composition
for sid in tqdm(unique_samples, desc='Per-patient clustering'):
    _, comp = per_patient_cluster(sid, cim_emb_pca, K_REPRO)
    if comp is not None:
        per_patient_comps[sid] = comp

valid_patients = list(per_patient_comps.keys())
print(f'Valid patients: {len(valid_patients)}')

In [ ]:
# ── Pairwise matching: collect JS divergence per matched cluster ──────────
# For each pair (A, B): match A's clusters to B's, record JS divergence per cluster.
# Shape: (k, n_pairs)
n_patients = len(valid_patients)
all_js     = [[] for _ in range(K_REPRO)]    # all_js[cluster_ref] = list of JS values across pairs

# Use patient 0 as the reference for visualisation; match all others to it
ref_sid    = valid_patients[0]
ref_comp   = per_patient_comps[ref_sid]

for sid in tqdm(valid_patients[1:], desc='Pairwise matching'):
    match_order, js_values = match_clusters(ref_comp, per_patient_comps[sid])
    for ref_cl, js in enumerate(js_values):
        all_js[ref_cl].append(float(js))

# Summary statistics
mean_js = [np.mean(v) for v in all_js]
print(f'\nMean JS divergence per cluster (reference = patient {ref_sid}):')
for cl, (m, vals) in enumerate(zip(mean_js, all_js)):
    dom = all_cell_types[ref_comp[cl].argmax()]
    print(f'  C{cl} ({dom:<22}): JS = {m:.3f} ± {np.std(vals):.3f}')
print(f'\nOverall mean JS: {np.mean([v for vals in all_js for v in vals]):.3f}')
print(f'(JS=0 → identical composition, JS=1 → maximally different)')

In [ ]:
# ── Plot 1: Boxplot of JS divergence per cluster ──────────────────────────
cmap_k = plt.cm.get_cmap('tab10', K_REPRO)

fig, ax = plt.subplots(figsize=(8, 4))
bp = ax.boxplot(all_js, patch_artist=True, widths=0.5)
for i, (patch, med) in enumerate(zip(bp['boxes'], bp['medians'])):
    patch.set_facecolor(cmap_k(i))
    patch.set_alpha(0.7)
    med.set_color('black')
    med.set_linewidth(2)
dom_labels = [f'C{i}\n({all_cell_types[ref_comp[i].argmax()][:12]})' for i in range(K_REPRO)]
ax.set_xticks(range(1, K_REPRO+1))
ax.set_xticklabels(dom_labels, fontsize=8)
ax.set(ylabel='Jensen-Shannon divergence', title=f'Cross-patient cluster reproducibility\nCIM + VICReg, MIBI_TNBC, k={K_REPRO}')
ax.set_ylim(0, 1)
ax.axhline(0.2, color='grey', linestyle='--', alpha=0.5, label='JS=0.2 reference')
ax.legend(fontsize=8)
plt.tight_layout()
plt.savefig(SAVE_DIR / 'reproducibility_boxplot_CIM.png', dpi=150, bbox_inches='tight')
plt.show()

# ── Plot 2: Heatmap — composition of matched clusters across patients ─────
# Align all patients to reference ordering, then plot composition heatmap
# Rows = patients, columns = matched cluster × cell type block
aligned_comps = [ref_comp]   # (n_patients, k, n_types)
for sid in valid_patients[1:]:
    order, _ = match_clusters(ref_comp, per_patient_comps[sid])
    aligned_comps.append(per_patient_comps[sid][order])

aligned_comps = np.stack(aligned_comps, axis=0)  # (n_patients, k, n_types)

# For each cluster, show the dominant cell-type fraction across patients
dom_type_per_cl = [all_cell_types[ref_comp[cl].argmax()] for cl in range(K_REPRO)]

fig, axes = plt.subplots(1, K_REPRO, figsize=(3*K_REPRO, 5), sharey=True)
for cl, ax in enumerate(axes):
    # Show full composition as stacked horizontal bar per patient
    comp_cl = aligned_comps[:, cl, :]   # (n_patients, n_types)
    ax.imshow(comp_cl, aspect='auto', vmin=0, vmax=comp_cl.max(), cmap='Blues')
    ax.set_title(f'C{cl}\n{dom_type_per_cl[cl][:14]}', fontsize=8)
    ax.set_xticks(range(len(all_cell_types)))
    ax.set_xticklabels([t[:6] for t in all_cell_types], rotation=90, fontsize=6)
    if cl == 0:
        ax.set_yticks(range(len(valid_patients)))
        ax.set_yticklabels(valid_patients, fontsize=6)
        ax.set_ylabel('Patient')
    ax.set_xlabel('Cell type', fontsize=7)

fig.suptitle(f'Matched cluster compositions across {len(valid_patients)} patients\nCIM + VICReg, MIBI_TNBC, k={K_REPRO}', fontsize=10)
plt.tight_layout()
plt.savefig(SAVE_DIR / 'reproducibility_heatmap_CIM.png', dpi=150, bbox_inches='tight')
plt.show()

---
## Experiment 2 — CIM vs EarlyFusion32: compartment distinctiveness

**Question:** Do CIM embeddings produce more biologically distinct tissue compartments than EarlyFusion32?

**Method:**
1. Cluster EarlyFusion32 embeddings (same k, same PCA dim)
2. For each model, compute **between-cluster JS divergence** — the mean JS divergence between all pairs of cluster composition vectors. Higher = clusters are more distinct from each other (better compartmentalisation).
3. Show side-by-side spatial maps for representative patients
4. Show side-by-side composition bars

**Expected result:** CIM produces more distinct compartments (higher between-cluster JS) because its sample-invariant embeddings cluster by biology rather than by staining batch.

In [ ]:
# PCA + cluster EarlyFusion32
pca_ef     = PCA(n_components=PCA_COMPONENTS, random_state=42)
ef_emb_pca = pca_ef.fit_transform(ef_emb.astype(np.float32))
print(f'EarlyFusion32 PCA explained variance: {pca_ef.explained_variance_ratio_.sum():.1%}')

km_ef      = MiniBatchKMeans(n_clusters=K_REPRO, random_state=42, n_init=10, batch_size=4096, max_iter=300)
ef_labels  = km_ef.fit_predict(ef_emb_pca)
print(f'EarlyFusion32 cluster sizes: {np.bincount(ef_labels).tolist()}')

_, ef_comp, _  = composition_from_labels(ef_labels, all_meta, patch_to_cell_annotations, all_cell_types, K_REPRO)
ef_comp_arr    = ef_comp   # (k, n_types)

In [ ]:
def between_cluster_js(comp):
    """Mean JS divergence between all pairs of cluster composition vectors."""
    k   = len(comp)
    js_vals = []
    for i in range(k):
        for j in range(i+1, k):
            p = comp[i] + 1e-9; p /= p.sum()
            q = comp[j] + 1e-9; q /= q.sum()
            js_vals.append(jensenshannon(p, q))
    return float(np.mean(js_vals)), js_vals


cim_bc_mean, cim_bc_vals = between_cluster_js(cim_comp_pooled)
ef_bc_mean,  ef_bc_vals  = between_cluster_js(ef_comp_arr)

print(f'Between-cluster JS divergence (k={K_REPRO}):')
print(f'  CIM + VICReg      : {cim_bc_mean:.3f}  (higher = more distinct compartments)')
print(f'  EarlyFusion32     : {ef_bc_mean:.3f}')
print(f'  Δ (CIM - EF)      : {cim_bc_mean - ef_bc_mean:+.3f}')

# Bar chart comparison
fig, axes = plt.subplots(1, 2, figsize=(10, 4))

ax = axes[0]
models = ['CIM + VICReg', 'EarlyFusion32']
means  = [cim_bc_mean, ef_bc_mean]
colors = ['steelblue', 'tomato']
bars   = ax.bar(models, means, color=colors, width=0.5)
ax.set_ylabel('Mean between-cluster JS divergence')
ax.set_title(f'Compartment distinctiveness (k={K_REPRO})\nMIBI_TNBC')
ax.set_ylim(0, max(means) * 1.3)
for bar, m in zip(bars, means):
    ax.text(bar.get_x() + bar.get_width()/2, m + 0.005, f'{m:.3f}', ha='center', va='bottom', fontsize=11)

ax = axes[1]
ax.boxplot([cim_bc_vals, ef_bc_vals], labels=models, patch_artist=True,
           boxprops=dict(facecolor='lightblue'), medianprops=dict(color='black', linewidth=2))
ax.set_ylabel('Pairwise cluster JS divergence')
ax.set_title(f'Distribution of pairwise JS (k={K_REPRO})')
plt.tight_layout()
plt.savefig(SAVE_DIR / 'compartment_distinctiveness.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Side-by-side spatial maps: CIM vs EarlyFusion32 ──────────────────────
cmap_k = plt.cm.get_cmap('tab10', K_REPRO)

fig, axes = plt.subplots(len(SHOW_PATIENTS), 2,
                         figsize=(14, 6*len(SHOW_PATIENTS)),
                         squeeze=False)

for row, sid in enumerate(SHOW_PATIENTS):
    H, W = sample_dims[sid]
    for col, (labels, model_name) in enumerate([
        (cim_labels_pooled, 'CIM + VICReg'),
        (ef_labels,         'EarlyFusion32 + VICReg'),
    ]):
        canvas = np.full((H, W, 4), [0.85, 0.85, 0.85, 1.0])
        for i in sample_to_meta_indices[sid]:
            _, y, x, _, _ = all_meta[i]
            canvas[y:y+ps, x:x+ps] = np.array(cmap_k(int(labels[i])))
        ax = axes[row, col]
        ax.imshow(canvas)
        ax.set_title(f'{model_name}\nPatient {sid}', fontsize=9)
        ax.axis('off')

handles = [mpatches.Patch(color=cmap_k(c), label=f'C{c}') for c in range(K_REPRO)]
fig.legend(handles=handles, loc='lower center', ncol=K_REPRO, frameon=False, fontsize=9)
fig.suptitle(f'CIM vs EarlyFusion32 — Spatial region maps, MIBI_TNBC (k={K_REPRO})', fontsize=12)
plt.tight_layout(rect=[0, 0.04, 1, 1])
plt.savefig(SAVE_DIR / 'spatial_comparison_CIM_vs_EF32.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Side-by-side composition bars: CIM vs EarlyFusion32 ──────────────────
fig, axes = plt.subplots(1, 2, figsize=(16, 5), sharey=False)

for ax, comp, title in [
    (axes[0], cim_comp_pooled, f'CIM + VICReg  (mean JS={cim_bc_mean:.3f})'),
    (axes[1], ef_comp_arr,     f'EarlyFusion32  (mean JS={ef_bc_mean:.3f})'),
]:
    df = pd.DataFrame(comp, columns=all_cell_types)
    df.plot(kind='bar', stacked=True, ax=ax, colormap='tab20', width=0.8, legend=False)
    ax.set(xlabel='Cluster', ylabel='Cell-type fraction', title=title)
    ax.set_xticklabels([f'C{c}' for c in range(K_REPRO)], rotation=0)

# Shared legend
handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, loc='lower center', ncol=6, frameon=False, fontsize=7,
           bbox_to_anchor=(0.5, -0.12))
fig.suptitle(f'Cluster composition comparison — MIBI_TNBC (k={K_REPRO})', fontsize=12)
plt.tight_layout()
plt.savefig(SAVE_DIR / 'composition_comparison_CIM_vs_EF32.png', dpi=150, bbox_inches='tight')
plt.show()

---
## Summary

Results saved to `z_RUNS/region_analysis_MIBI_TNBC/ps{REGION_PATCH_SIZE}/`:

| File | Content |
|------|--------|
| `k_sweep_CIM.png` | Elbow + silhouette sweep |
| `k_{k}/umap_CIM.png` | UMAP coloured by region cluster |
| `k_{k}/spatial_map_CIM.png` | Spatial tissue maps (representative patients) |
| `k_{k}/composition_CIM.png` | Cell-type composition per cluster |
| `reproducibility_boxplot_CIM.png` | **Exp. 1** — JS divergence per cluster across patients |
| `reproducibility_heatmap_CIM.png` | **Exp. 1** — Per-patient matched composition heatmap |
| `compartment_distinctiveness.png` | **Exp. 2** — Mean between-cluster JS: CIM vs EF32 |
| `spatial_comparison_CIM_vs_EF32.png` | **Exp. 2** — Side-by-side spatial maps |
| `composition_comparison_CIM_vs_EF32.png` | **Exp. 2** — Side-by-side composition bars |

In [ ]:
print('=== SUMMARY ===')
print(f'Dataset          : MIBI_TNBC')
print(f'Patients         : {len(valid_patients)}')
print(f'Total patches    : {len(all_meta):,}')
print(f'Patch size       : {REGION_PATCH_SIZE}px')
print(f'k (analysis)     : {K_REPRO}')
print()
print('--- Experiment 1: Cross-patient reproducibility ---')
print(f'Overall mean JS  : {np.mean([v for vals in all_js for v in vals]):.3f}')
for cl in range(K_REPRO):
    dom = all_cell_types[ref_comp[cl].argmax()]
    print(f'  C{cl} ({dom:<20}): {np.mean(all_js[cl]):.3f} ± {np.std(all_js[cl]):.3f}')
print()
print('--- Experiment 2: CIM vs EarlyFusion32 ---')
print(f'CIM between-cluster JS : {cim_bc_mean:.3f}')
print(f'EF32 between-cluster JS: {ef_bc_mean:.3f}')
print(f'Δ (CIM - EF32)         : {cim_bc_mean - ef_bc_mean:+.3f}')